
![DBAcademy](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/databricks_academy.png)

# Lecture - Ensure Data Quality with Expectations

## Overview

In this lecture, you will learn how to create and add data quality expectations to your pipelines, apply expectations to validate streaming data, and understand the available actions for handling records that fail data quality checks.

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Explain what expectations are** and how they are used to enforce data quality rules in Apache Spark™ Declarative Pipelines
2. **Write the CONSTRAINT syntax** to define data quality expectations on streaming tables
3. **Describe the three violation actions** — WARN, DROP, and FAIL — and explain when to use each
4. **Apply expectations to a streaming table** using SQL with all three violation actions
5. **Trace how a row of data is evaluated** through expectations in the pipeline and explain what happens when a constraint is violated


## A. What Are Expectations

<br>

<div style="max-width:1000px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
  <div style="display:flex; gap:24px; align-items:flex-start; flex-wrap:wrap;">
    <!-- Column 1 -->
    <div style="flex:1 1 0; min-width:260px;">
      Apache Spark™ Declarative Pipelines lets you apply <b>data quality rules</b> called <b>expectations</b> during your ETL processes. 
      
  These rules validate your data row-by-row to ensure integrity.
    </div>
    <!-- Column 2 (empty placeholder for now) -->
    <div style="flex:1 1 0; min-width:260px;">
      <div style="background:#F9F7F4; border:2px solid #EEEDE9; border-radius:10px; padding:16px 18px;">
        <div style="font-size:9pt; font-weight:800; color:#618794; text-transform:uppercase; letter-spacing:0.06em; margin-bottom:10px;">
          General Syntax
        </div>
<div class="code-block" data-language="sql">
CONSTRAINT constraint_name
EXPECT (column_condition)
[ON VIOLATION action]
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
  document.querySelectorAll('.code-block').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var lang = block.getAttribute('data-language') || 'sql';
    var code = block.textContent.trim();
    var id = 'code-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML = 
      '<div style="position:relative;margin:16px 0;">' +
        '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
        '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
      '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
      var t = document.createElement('textarea');
      t.value = code;
      document.body.appendChild(t);
      t.select();
      document.execCommand('copy');
      document.body.removeChild(t);
      this.textContent = '✓ Copied!';
      setTimeout(() => this.textContent = 'Copy', 2000);
    };
  });
})();
</script>
      </div>
    </div>
  </div>
</div>


## B. The Three Violation Actions

##### Click each tab below to explore the syntax and use case of the three violation actions.

<div style="max-width: 900px; margin: 0 auto; font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

  <style>
    /* Card layout (mirrors Batch Ingestion style) */
    .exp-grid {
      display: flex;
      flex-direction: column;
      gap: 40px;
      justify-content: center;
      align-items: center;
      margin-top: 16px;
    }
    .exp-box {
      width: 100%;
      background: #F9F7F4;
      border-radius: 8px;
      box-shadow: 0 2px 8px rgba(27,49,57,0.06);
      padding: 16px 18px 18px 18px;
      box-sizing: border-box;
      position: relative;
      overflow: hidden;
      display: flex;
      flex-direction: column;
      gap: 12px;
    }
    .exp-box::before {
      content: "";
      position: absolute;
      top: 0; left: 0;
      width: 100%; height: 6px;
    }
    .exp-box.warn::before  { background: #FFAB00; }
    .exp-box.drop::before  { background: #FF5F46; }
    .exp-box.fail::before  { background: #98102A; }

    .exp-box-header {
      display: flex;
      align-items: center;
      justify-content: space-between;
    }
    .exp-box-title {
      font-size: 16pt;
      font-weight: 700;
      color: #0B2026;
    }
    .exp-box-subtitle {
      font-size: 14pt;
      color: #64748b;
      margin-top: 2px;
    }
    .exp-pane {
      display: flex;
      flex-direction: column;
      gap: 10px;
    }
    .exp-sql-box {
      background:#FDFDFD;
      border-radius:8px;
      border:1px solid #E0E4E7;
      padding:10px 12px;
    }
    .exp-sql-title {
      font-size:14px;
      font-weight:700;
      color:#1B3139;
      margin-bottom:4px;
      text-transform:uppercase;
      letter-spacing:0.06em;
    }
    .exp-code {
      background:#FFFFFF;
      border-radius:6px;
      border:1px solid #E0E4E7;
      padding:10px 12px;
      font-family:Consolas, Monaco, 'Courier New', monospace;
      font-size:14px;
      line-height:1.7;
      color:#1B3139;
      overflow-x:auto;
      margin:0;
    }

    .exp-result-box {
      background:#FFFFFF;
      border-radius:8px;
      border:1px solid #E0E4E7;
      padding:10px 14px;
      font-size:14px;
      color:#0B2026;
      line-height:1.7;
    }
    .exp-result-box ul {
      margin:6px 0 0 1.1em;
      padding:0;
    }

    /* Tabs (same pattern as Batch carousel) */
    .expect-tabbar {
      display: flex;
      border-bottom: 2px solid #EEEDE9;
      margin-bottom: 0;
    }
    .expect-tab {
      padding: 10px 18px;
      border: none;
      border-bottom: 3px solid transparent;
      background: none;
      font-size: 14pt;
      font-weight: bold;
      color: #888;
      cursor: pointer;
      margin-bottom: -2px;
    }

    .expect-panel { display:none; }
  </style>

  <!-- Tabs -->
  <div class="expect-tabbar">
    <button class="expect-tab" onclick="showExpectTab(1)">WARN</button>
    <button class="expect-tab" onclick="showExpectTab(2)">DROP</button>
    <button class="expect-tab" onclick="showExpectTab(3)">FAIL</button>
  </div>
  <!-- WARN panel -->
  <div class="expect-panel">
    <div class="exp-grid">
      <div class="exp-box warn">
        <div class="exp-box-header">
          <div>
            <div class="exp-box-title">WARN — Log Violations, Keep Rows</div>
            <div class="exp-box-subtitle">Default behavior when no action is specified</div>
          </div>
        </div>
        <div class="exp-pane">
          <div class="exp-sql-box">
            <div class="exp-sql-title">SQL Syntax Example</div>
<div class="code-block" data-language="sql">
CONSTRAINT valid_notification
EXPECT (notifications IN ('Y','N'))
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
  document.querySelectorAll('.code-block').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var lang = block.getAttribute('data-language') || 'sql';
    var code = block.textContent.trim();
    var id = 'code-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML =
      '<div style="position:relative;margin:16px 0;">' +
        '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
        '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
      '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
      var t = document.createElement('textarea');
      t.value = code;
      document.body.appendChild(t);
      t.select();
      document.execCommand('copy');
      document.body.removeChild(t);
      this.textContent = '✓ Copied!';
      setTimeout(() => this.textContent = 'Copy', 2000);
    };
  });
})();
</script>
          </div>
          <div class="exp-result-box">
            <strong style="color:#FFAB00;">Result</strong>
            <ul>
              <li>Invalid rows are still written to the target.</li>
              <li>Logs include counts of valid vs. invalid records and other metrics.</li>
            </ul>
          </div>
        </div>
      </div>
    </div>
  </div>
  <!-- DROP panel -->
  <div class="expect-panel">
    <div class="exp-grid">
      <div class="exp-box drop">
        <div class="exp-box-header">
          <div>
            <div class="exp-box-title">DROP — Exclude Invalid Rows</div>
            <div class="exp-box-subtitle">Remove bad records from the target table</div>
          </div>
        </div>
        <div class="exp-pane">
          <div class="exp-sql-box">
            <div class="exp-sql-title">SQL Syntax Example</div>
<div class="code-block" data-language="sql">
CONSTRAINT valid_date
EXPECT (order_timestamp > "2021-01-01")
ON VIOLATION DROP ROW
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
  document.querySelectorAll('.code-block').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var lang = block.getAttribute('data-language') || 'sql';
    var code = block.textContent.trim();
    var id = 'code-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML =
      '<div style="position:relative;margin:16px 0;">' +
        '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
        '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
      '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
      var t = document.createElement('textarea');
      t.value = code;
      document.body.appendChild(t);
      t.select();
      document.execCommand('copy');
      document.body.removeChild(t);
      this.textContent = '✓ Copied!';
      setTimeout(() => this.textContent = 'Copy', 2000);
    };
  });
})();
</script>
          </div>
          <div class="exp-result-box">
            <strong style="color:#FF5F46;">Result</strong>
            <ul>
              <li>Invalid rows are dropped from the table.</li>
              <li>The count of dropped rows is logged alongside other metrics.</li>
            </ul>
          </div>
        </div>
      </div>
    </div>
  </div>
  <!-- FAIL panel -->
  <div class="expect-panel">
    <div class="exp-grid">
      <div class="exp-box fail">
        <div class="exp-box-header">
          <div>
            <div class="exp-box-title">FAIL — Stop the Flow on Violations</div>
            <div class="exp-box-subtitle">Fail a specific flow when constraints are broken</div>
          </div>
        </div>
        <div class="exp-pane">
          <div class="exp-sql-box">
            <div class="exp-sql-title">SQL Syntax Example</div>
<div class="code-block" data-language="sql">
CONSTRAINT valid_id
EXPECT (customer_id IS NOT NULL)
ON VIOLATION FAIL UPDATE
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
  document.querySelectorAll('.code-block').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var lang = block.getAttribute('data-language') || 'sql';
    var code = block.textContent.trim();
    var id = 'code-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML =
      '<div style="position:relative;margin:16px 0;">' +
        '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
        '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
      '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
      var t = document.createElement('textarea');
      t.value = code;
      document.body.appendChild(t);
      t.select();
      document.execCommand('copy');
      document.body.removeChild(t);
      this.textContent = '✓ Copied!';
      setTimeout(() => this.textContent = 'Copy', 2000);
    };
  });
})();
</script>
          </div>
          <div class="exp-result-box">
            <strong style="color:#98102A;">Result</strong>
            <ul>
              <li>Causes a failure of a single flow and does not cause other flows in your pipeline to fail.</li>
              <li>Manual intervention is required to resolve the issue.</li>
            </ul>
          </div>
        </div>
      </div>
    </div>
  </div>
</div>
<script>
function showExpectTab(n) {
  var tabs   = document.getElementsByClassName("expect-tab");
  var panels = document.getElementsByClassName("expect-panel");
  var colors = ["#FFAB00", "#FF5F46", "#98102A"];
  for (var i = 0; i < tabs.length; i++) {
    tabs[i].style.color = "#888";
    tabs[i].style.borderBottom = "3px solid transparent";
    panels[i].style.display = "none";
  }
  tabs[n-1].style.color = colors[n-1];
  tabs[n-1].style.borderBottom = "3px solid " + colors[n-1];
  panels[n-1].style.display = "block";
}
// default to WARN
window.addEventListener("DOMContentLoaded", function(){ showExpectTab(1); });
</script>


## C. Adding Expectations — Full SQL Example

Let's look at an example of how to add constraints in Declarative Pipelines. Here we will create a streaming table named <code>orders_silver</code> with all three constraint types applied.

##### Click each step below to explore how the constraints are added.
<br>

<div style="width:100%; font-family:'Segoe UI',sans-serif; max-width:1000px; margin:0 auto;">

<style>
.journey-wrap8 { display:flex; flex-direction:column; gap:0; position:relative; }
.journey-wrap8::before {
  content:""; position:absolute; left:27px; top:44px; bottom:44px;
  width:2px; background:linear-gradient(to bottom, #2272B4, #FFAB00, #FF5F46, #98102A);
  z-index:0;
}
.step-row8 { display:flex; align-items:flex-start; gap:18px; cursor:pointer; position:relative; z-index:1; margin-bottom:6px; }
.step-badge8 {
  width:56px; height:56px; border-radius:50%; display:flex; align-items:center; justify-content:center;
  font-size:20px; font-weight:800; color:white; flex-shrink:0; border:3px solid white;
  box-shadow:0 2px 10px rgba(0,0,0,0.15); transition:transform 0.2s, box-shadow 0.2s; z-index:2; position:relative;
}
.step-row8:hover .step-badge8 { transform:scale(1.1); box-shadow:0 6px 18px rgba(0,0,0,0.2); }
.step-card8 { flex:1; background:#F9F7F4; border-radius:10px; border:1.5px solid #EEEDE9; overflow:hidden; transition:box-shadow 0.25s, border-color 0.25s; }
.step-card8:hover { box-shadow:0 4px 18px rgba(0,0,0,0.10); }
.step-card8.active { border-color:var(--sc8); box-shadow:0 4px 18px rgba(0,0,0,0.12); }
.step-header8 { display:flex; align-items:center; justify-content:space-between; padding:14px 18px; background:#F9F7F4; }
.step-header8-left { display:flex; align-items:center; gap:10px; }
.step-dot8 { width:10px; height:10px; border-radius:50%; background:var(--sc8); flex-shrink:0; }
.step-title8 { font-size:13pt; font-weight:700; color:#0B2026; }
.step-subtitle8 { font-size:11pt; color:#618794; margin-top:1px; }
.step-chevron8 { font-size:14px; color:#90A5B1; transition:transform 0.25s; user-select:none; }
.step-card8.active .step-chevron8 { transform:rotate(180deg); color:var(--sc8); }
.step-body8 { display:none; padding:0 18px 18px 18px; border-top:1.5px solid #EEEDE9; animation:expandIn8 0.25s ease; }
@keyframes expandIn8 { from{opacity:0;transform:translateY(-8px);} to{opacity:1;transform:translateY(0);} }
.step-card8.active .step-body8 { display:block; }
.diag-box8 { background:white; border-radius:8px; border:1.5px solid #DCE0E2; padding:14px 16px; margin-top:12px; }
.step-desc8 { font-size:11pt; color:#1B3139; line-height:1.8; margin-top:12px; }
.step-desc8 strong { color:#0B2026; }
</style>

<div class="journey-wrap8">

  <!-- STEP 1 -->
  <div class="step-row8" onclick="toggleStep8('c1')">
    <div class="step-badge8" style="background:#2272B4;">1</div>
    <div class="step-card8" id="c1" style="--sc8:#2272B4;">
      <div class="step-header8">
        <div class="step-header8-left">
          <div class="step-dot8"></div>
          <div>
            <div class="step-title8">Define the Streaming Table</div>
            <div class="step-subtitle8">CREATE OR REFRESH STREAMING TABLE with constraint block</div>
          </div>
        </div>
        <div class="step-chevron8">▼</div>
      </div>
      <div class="step-body8">
        <div class="diag-box8">
          <div class="code-block" data-language="sql">
CREATE OR REFRESH STREAMING TABLE 2_silver_db.orders_silver
 (
   -- constraints go here
 )
AS
          </div>
        </div>
      </div>
    </div>
  </div>

  <!-- STEP 2: WARN -->
  <div class="step-row8" onclick="toggleStep8('c2')">
    <div class="step-badge8" style="background:#FFAB00;">2</div>
    <div class="step-card8" id="c2" style="--sc8:#FFAB00;">
      <div class="step-header8">
        <div class="step-header8-left">
          <div class="step-dot8"></div>
          <div>
            <div class="step-title8">WARN Constraint — valid_notifications</div>
            <div class="step-subtitle8">WARN in metrics</div>
          </div>
        </div>
        <div class="step-chevron8">▼</div>
      </div>
      <div class="step-body8">
        <div class="diag-box8">
          <div class="code-block" data-language="sql">
CONSTRAINT valid_notifications
EXPECT (notifications IN ('Y','N')),
          </div>
        </div>
      </div>
    </div>
  </div>

  <!-- STEP 3: FAIL -->
  <div class="step-row8" onclick="toggleStep8('c3')">
    <div class="step-badge8" style="background:#FF5F46;">3</div>
    <div class="step-card8" id="c3" style="--sc8:#FF5F46;">
      <div class="step-header8">
        <div class="step-header8-left">
          <div class="step-dot8"></div>
          <div>
            <div class="step-title8">FAIL Constraint — valid_date</div>
            <div class="step-subtitle8">FAIL the pipeline</div>
          </div>
        </div>
        <div class="step-chevron8">▼</div>
      </div>
      <div class="step-body8">
        <div class="diag-box8">
          <div class="code-block" data-language="sql">
CONSTRAINT valid_date
EXPECT (order_timestamp > "2021-01-01")
ON VIOLATION FAIL UPDATE,
          </div>
        </div>
      </div>
    </div>
  </div>

  <!-- STEP 4: DROP -->
  <div class="step-row8" onclick="toggleStep8('c4')">
    <div class="step-badge8" style="background:#98102A;">4</div>
    <div class="step-card8" id="c4" style="--sc8:#98102A;">
      <div class="step-header8">
        <div class="step-header8-left">
          <div class="step-dot8"></div>
          <div>
            <div class="step-title8">DROP Constraint — valid_id</div>
            <div class="step-subtitle8">DROP the row</div>
          </div>
        </div>
        <div class="step-chevron8">▼</div>
      </div>
      <div class="step-body8">
        <div class="diag-box8">
          <div class="code-block" data-language="sql">
CONSTRAINT valid_id
EXPECT (customer_id IS NOT NULL)
ON VIOLATION DROP ROW
          </div>
        </div>
      </div>
    </div>
  </div>

</div>

<!-- RESULT -->
<div style="margin-top:20px; padding:18px 24px; background:#F9F7F4; border:3px solid #FF5F46; border-radius:10px;">
  <div style="font-weight:700; margin-bottom:8px; font-size:14pt; color:#0B2026;">
    <b>RESULT</b>: <code>orders_silver</code> with all three data quality constraints
  </div>

  <div class="code-block" data-language="sql">
CREATE OR REFRESH STREAMING TABLE 2_silver_db.orders_silver
 (
   CONSTRAINT valid_notifications EXPECT (notifications IN ('Y','N')),
   CONSTRAINT valid_date EXPECT (order_timestamp > "2021-01-01") ON VIOLATION FAIL UPDATE,
   CONSTRAINT valid_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW
 )
AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp,
  customer_id,
  notifications
FROM STREAM 1_bronze_db.orders_bronze;
  </div>
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
  // Copy + light-background code-blocks
  document.querySelectorAll('.code-block').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var lang = block.getAttribute('data-language') || 'sql';
    var code = block.textContent.trim();
    var id = 'code-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML =
      '<div style="position:relative;margin:16px 0;">' +
        '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
        '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;">' +
          '<code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code>' +
        '</pre>' +
      '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
      var t = document.createElement('textarea');
      t.value = code;
      document.body.appendChild(t);
      t.select();
      document.execCommand('copy');
      document.body.removeChild(t);
      this.textContent = '✓ Copied!';
      setTimeout(() => this.textContent = 'Copy', 2000);
    };
  });
})();

function toggleStep8(id) {
  var card = document.getElementById(id);
  var isActive = card.classList.contains("active");
  document.querySelectorAll(".step-card8").forEach(function(c){ c.classList.remove("active"); });
  if (!isActive) { card.classList.add("active"); }
}
window.addEventListener("DOMContentLoaded", function(){ toggleStep8("c1"); });
</script>
</div>


## D. Actions Overview — How Expectations Work in the Pipeline

<div style="text-align: center; margin-top: 20px;">
  <img
    src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.2/images/20261007T191216Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_ensure_data_quality/actions_overview.png"
    alt="LakeFlow Connect Unified Ingestion"
    style="width: 900px; max-width: 100%; height: auto;">
</div>


<!-- Q2 2025 Note -->
<div style="border-left:4px solid #ff9800; background:#fff3e0; padding:14px 18px; border-radius:4px; margin:0;">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size:1.1em;">Important — As of Q2 2025</strong>
  <div style="color:#333; font-size:10.5pt; line-height:1.75;">
    Materialized views that use <b>expectations</b> are <strong>always fully refreshed</strong> during pipeline runs.
  </div>
</div>
</div>

#####EXPAND FOR ADDITIONAL NOTES
<details>

Now let’s look at how expectations are applied within the pipeline architecture.

Here’s what happens step by step:

1. A row of data enters the pipeline.

2. That row is evaluated against any defined expectations (constraints).

- If the row passes all expectations, it’s kept, and the pipeline continues processing as normal.

- If the row fails an expectation, the action defined in the constraint determines what happens next:
<ul>
<li>WARN (default): The failure is logged, the row is still included, and the pipeline continues.
<li>DROP: The row is discarded, but the pipeline continues processing remaining data.
<li>FAIL: The pipeline fails immediately for that specific flow, and manual intervention is required. Other flows are unaffected.
</li>
</ul>

As of Q2 2025, any materialized views that use expectations will always be fully refreshed during pipeline runs.

</details>


## E. Conclusion

In this lecture, you learned how to use expectations to enforce data quality within Apache Spark™ Declarative Pipelines:

1. **Expectations are data quality rules** defined using the `CONSTRAINT ... EXPECT (...)` syntax directly within your pipeline SQL — validating data row-by-row during ETL.
2. **WARN** (the default action) logs violations and still writes invalid rows to the target — ideal for non-critical quality monitoring without interrupting the pipeline.
3. **DROP** removes invalid rows from the output and logs the count — suitable for cases where bad data must be excluded but the pipeline should continue running.
4. **FAIL** stops the specific flow when a violation occurs and requires manual intervention — used for critical data issues that must be resolved before downstream processing.
5. **Multiple constraints** can coexist in a single table definition, each evaluated independently — giving you layered, granular control over data quality across all columns.

### Next Steps

In the next section, you will explore hands-on demonstrations of data quality expectations in action — running pipelines with constraints, observing metrics in the execution insights panel, and understanding how violations are surfaced within the Spark Declarative Pipelines UI.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/>
<a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> |
<a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> |
<a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
